In [1]:
import Tensor as t
import numpy as np
import matplotlib.pyplot as plt
import Operations as o
import Compile as c
from sklearn.datasets import fetch_openml

We will stick to the row major order to align with numpy. This means our "dense" layers will be

$$\bold{Y} = \bold{X}\bold{W} + \bold{b}$$

Where $\bold{X}$ is the row vector in question. Might implement a technique called batching in the future.

In [2]:
# Fetch the MNIST dataset (this might take a minute to download)
mnist = fetch_openml('mnist_784', version=1, as_frame=False)

# Split into features (images) and labels
X, y = mnist["data"], mnist["target"]

print(f"Dataset shape: {X.shape}")

Dataset shape: (70000, 784)


In [3]:
def encode(val):
    z = np.zeros(10,dtype=np.float64)
    z[int(val)] += 1.0
    return z

y_cleaned = np.array([encode(k) for k in y])

x_cleaned = X.astype(np.float64)

In [4]:
print(x_cleaned.shape)

(70000, 784)


In [5]:
print(y_cleaned.shape)

(70000, 10)


In [6]:
#current architecture: Dense(784 15) sAct softmax Dense(15 10) sAct softmax (done!)
#paramaters
w_1 = np.random.random_sample(size=(784, 15))
b_1 = np.random.random_sample(size=(1,15))

w_2 = np.random.random_sample(size=(15,10))
b_2 = np.random.random_sample(size=(1,10))

def pipeline(input):
    L_1 = (o.copy(t.TensorNode(input,is_param=False))) @ t.TensorNode(w_1) + t.TensorNode(b_1)
    L_1 = o.sAct(L_1)/(o.sum(L_1,axis=(1),keepDim=True,epsilon=1e-7))
    
    L_2 = L_1 @ t.TensorNode(w_2) + t.TensorNode(b_2)

    L_2 = o.sAct(L_2)/(o.sum(L_2,axis=(1),keepDim=True,epsilon=1e-7))
    return L_2


In [7]:
L_1 = (o.copy(t.TensorNode(x_cleaned[0],is_param=False))) @ t.TensorNode(w_1) + t.TensorNode(b_1)
L_1 = o.regularization(o.sAct(L_1))

L_2 = L_1 @ t.TensorNode(w_2) + t.TensorNode(b_2)

L_2 = o.regularization(o.sAct(L_2))

In [8]:
modelCompiler = c.Pipeline(L_2.compile())
#use model to actually get predictions and vector ouputs!
print(L_2.data)
print(y_cleaned[0])

[[0.11023681 0.10129714 0.09142731 0.12081903 0.0995994  0.09799425
  0.10618657 0.07568373 0.09886759 0.09788815]]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]


In [12]:
modelCompiler.update_input(x_cleaned[0])
print(L_2.data)

TypeError: TensorNode.__init__.<locals>.dummy_update() missing 1 required positional argument: 'gradient'

In [9]:
loss = o.norm_squared(pipeline(x_cleaned) - t.TensorNode(y_cleaned,is_param=False))

In [35]:
lossTrainer = c.Pipeline(loss.compile())
print(len(lossTrainer.rev_topo_sort))
for j in range(10):
    
    print(loss.data) # Mean Square error Currently

    lossTrainer.train()
    lossTrainer.update(0.1)

    print(loss.data) # Mean Square Error Afterwards

12
62870.65253640729
62851.68281512268
62851.68281512268
62828.70148638968
62828.70148638968
62800.3401193374
62800.3401193374
62764.82663544824
62764.82663544824
62718.50599748338
62718.50599748338
62652.47654376022
62652.47654376022
62543.0496326273
62543.0496326273
62256.48822838556
62256.48822838556
140523.76725864224
140523.76725864224
62999.705557602836


WE DID IT. Below is the accuracy:

In [11]:
correct = 0
for j in range(50_000):
    modelCompiler.update_input(np.array([x_cleaned[j]]))
    if(str(np.argmax(model.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/50_000))

print("test dataset:")
correct = 0
for j in range(50_001,70_000):
    modelCompiler.update_input(x_cleaned[j])
    if(str(np.argmax(model.data)) == y[j]):
        correct+=1

print("Raw correct: " + str(correct))
print("accruacy " + str(correct/20_000))


TypeError: TensorNode.__init__.<locals>.dummy_update() missing 1 required positional argument: 'gradient'

Check out data.npz to import paramaters